# Датасет по модели численности (resist_id=39)

Цель: собрать датафрейм из CSV (raw) со столбцами:
- id
- month
- year
- operation_id
- fact_value
- fact_predict
- predict_new (пока пустое)
- workDayMonth_2


In [ ]:
import json
from pathlib import Path

import pandas as pd

RAW_DIR = Path("../data/raw")
RAW_FILES = [
    RAW_DIR / "data-2022.csv",
    RAW_DIR / "data-2023.csv",
    RAW_DIR / "data-2024.csv",
    RAW_DIR / "data-2025.csv",
]

RAW_FILES

In [ ]:
# Загружаем все CSV и фильтруем resist_id=39

dfs = []
for p in RAW_FILES:
    df_part = pd.read_csv(p, dtype=str)
    dfs.append(df_part)

df = pd.concat(dfs, ignore_index=True)

df["period"] = pd.to_datetime(df["period"], errors="coerce")
df["resist_id"] = pd.to_numeric(df["resist_id"], errors="coerce").astype("Int64")

df_39 = df[df["resist_id"] == 39].copy()

print("Всего строк resist_id=39:", len(df_39))
df_39.head()

In [ ]:
# Data Cleaning: фильтруем только утвержденные записи до парсинга JSON

# Нормализация булевых полей

def to_bool(series: pd.Series) -> pd.Series:
    return series.astype(str).str.lower().isin(["true", "t", "1"])

bool_cols = ["send", "validate_one", "validate_two", "cancel_approve", "forecast"]
for col in bool_cols:
    if col in df_39.columns:
        df_39[col] = to_bool(df_39[col])

# Валидация этапов

df_39["validate_count"] = pd.to_numeric(df_39["validate_count"], errors="coerce").fillna(0).astype(int)

mask_sent = df_39["send"]
mask_not_cancelled = ~df_39["cancel_approve"]
mask_approved = (
    (df_39["validate_count"] == 0)
    | ((df_39["validate_count"] == 1) & df_39["validate_one"])
    | ((df_39["validate_count"] >= 2) & df_39["validate_one"] & df_39["validate_two"])
)

# Исключение авто-прогнозов (Data Leakage / Loop prevention)

PREDICT_COMMENT = "first_predict_value_record"
if "comment" in df_39.columns:
    df_39["comment"] = df_39["comment"].fillna("")
    mask_predict_record = df_39["comment"] == PREDICT_COMMENT
else:
    mask_predict_record = pd.Series(False, index=df_39.index)

# Итоговый фильтр (важно: forecast не фильтруем!)

df_ok = df_39[mask_sent & mask_not_cancelled & mask_approved & ~mask_predict_record].copy()

# Разбор operation_parameter + workDayMonth_2

def extract_workdaymonth(calc_json: str):
    if pd.isna(calc_json):
        return None
    try:
        obj = json.loads(calc_json)
    except Exception:
        return None
    v = obj.get("workDayMonth_2")
    if isinstance(v, dict):
        return v.get("value")
    return None


records = []
for _, row in df_ok.iterrows():
    workday = extract_workdaymonth(row.get("calculate_parameter"))
    op_raw = row.get("operation_parameter")
    if pd.isna(op_raw):
        continue
    try:
        op_data = json.loads(op_raw)
    except Exception:
        continue

    for op_id, payload in op_data.items():
        if not isinstance(payload, dict):
            continue
        period = row.get("period")
        if pd.isna(period):
            continue
        records.append({
            "id": row.get("id"),
            "month": int(period.month),
            "year": int(period.year),
            "operation_id": op_id,
            "fact_value": payload.get("fact_value"),
            "fact_predict": payload.get("fact_predict"),
            "predict_new": None,
            "workDayMonth_2": workday,
        })

df_model39 = pd.DataFrame(records)

# Типы и базовые поля

df_model39["fact_value"] = pd.to_numeric(df_model39["fact_value"], errors="coerce")
df_model39["fact_predict"] = pd.to_numeric(df_model39["fact_predict"], errors="coerce")
df_model39["workDayMonth_2"] = pd.to_numeric(df_model39["workDayMonth_2"], errors="coerce")

# MAPE по текущему прогнозу (fact_predict vs fact_value)
with pd.option_context("mode.use_inf_as_na", True):
    df_model39["mape"] = (
        (df_model39["fact_value"] - df_model39["fact_predict"]).abs()
        / df_model39["fact_value"].abs()
    )

# Правила для fact_value == 0:
# - если fact_predict == 0 => MAPE = 0 (0%)
# - если fact_predict != 0 => MAPE = 1 (100%)
mask_zero = df_model39["fact_value"] == 0
mask_zero_pred = mask_zero & (df_model39["fact_predict"] == 0)
mask_zero_nonzero_pred = mask_zero & (df_model39["fact_predict"] != 0)

df_model39.loc[mask_zero_pred, "mape"] = 0.0
df_model39.loc[mask_zero_nonzero_pred, "mape"] = 1.0

# Feature Engineering (Time Series)

# дата из года и месяца

df_model39["date"] = pd.to_datetime(
    dict(year=df_model39["year"], month=df_model39["month"], day=1),
    errors="coerce",
)

# сортировка обязательна для лагов

df_model39 = df_model39.sort_values(["operation_id", "year", "month"]).reset_index(drop=True)

# лаги и скользящие средние (по operation_id)

group = df_model39.groupby("operation_id")

df_model39["lag_1"] = group["fact_value"].shift(1)

df_model39["lag_12"] = group["fact_value"].shift(12)

df_model39["roll_mean_3"] = group["fact_value"].transform(
    lambda s: s.shift(1).rolling(3, min_periods=3).mean()
)

df_model39["roll_mean_6"] = group["fact_value"].transform(
    lambda s: s.shift(1).rolling(6, min_periods=6).mean()
)

# тренд: отношение прошлого месяца к среднему за 3 мес

df_model39["trend"] = df_model39["lag_1"] / df_model39["roll_mean_3"]

# Финальная очистка

# 1) удаляем строки без факта

df_model39 = df_model39.dropna(subset=["fact_value"])

# 2) удаляем первые 12 месяцев (лаг 12)

df_model39 = df_model39.dropna(subset=["lag_12"])

# 3) оставшиеся пропуски заполняем нулями

df_model39 = df_model39.fillna(0)

print("Итоговых строк:", len(df_model39))
df_model39.head()